# Model A-freq v3: Student-t quality-difference pilot

This standalone Colab notebook implements the latest iteration from “Diagnostica gate 5.1”.
The earlier v2 used a stabilized log-ratio. This v3 instead fits
`d_quality = q_lock - q_control` with a Student-t(4) likelihood and non-centred,
sum-to-zero effects for configurations, harmonic frequencies and backgrounds.

Start with `RUN_MODE = "PILOT"`. After a PILOT PASS, change only that setting to
`"FULL"`, reconnect to a clean runtime and run all again. PILOT is non-reportable.
Its fit uses 4 chains, 1,000 tuning steps, 1,000 retained draws and `target_accept=0.95`.

### Scientific definition

- Tone peak amplitude / background standard deviation: `TONE_SNR = 1.25`.
- Localisation quality: `q = exp(-log(2) * (frequency_error / 1 Hz)**2)`.
- The 64-point forecast response is `forecast(background + tone) - forecast(background)`.
- Estimate frequency by mean removal, periodic Hann window and a 4096-point FFT,
  taking the global argmax in 2–250 Hz without rounding. Rectangular is the sensitivity window.
- Each pair shares configuration, generator, background and phase. One tone is at a lock;
  the matched non-lock control alternates below/above the lock across replicates.
- `d_quality` lies in [-1, 1]; negative values mean worse localisation at locks.
  `beta_bar` is the model's population location on this difference scale.
- Support requires `P(beta_bar <= -0.20) >= 0.95`; practical equivalence requires
  `P(abs(beta_bar) <= 0.10) >= 0.95`. These are **absolute quality units**, not relative percentages.
- Student-t is a working continuous likelihood with unbounded support. Many nearly zero
  differences can still cause poor mixing. This revision is a candidate to test, not a guarantee
  of convergence or of a scientifically valid fit. All existing validity gates remain required.

Compatible v3/v2/v1 pair tables on Drive are reused only after manifest, design and hash checks.
No old posterior is reused. New results use the separate `model_A_freq_snr125_v3_fixed1` namespace.
Synthetic recovery and local smoke output are method checks, never empirical evidence.


## 1. Setup

### 1.1 Locate or clone the repository

The repository supplies the frozen model registry, retrained checkpoints, synthetic generators,
and atomic checkpoint helpers.


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/FedericoSabbadini/patchAliasing.git"
MARKER = Path("chronos") / "bayesian" / "probe_lib.py"


def on_colab() -> bool:
    try:
        return importlib.util.find_spec("google.colab") is not None
    except ModuleNotFoundError:
        return False


IS_COLAB = on_colab()


def find_repo() -> Path:
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / MARKER).is_file():
            return candidate
    target = Path("/content/patchAliasing") if IS_COLAB else here / "patchAliasing"
    if not (target / MARKER).is_file():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(target)])
    if not (target / MARKER).is_file():
        raise FileNotFoundError(f"{MARKER} is missing from {target}")
    return target


REPO = find_repo()
BAYES_DIR = REPO / "chronos" / "bayesian"
if str(BAYES_DIR) not in sys.path:
    sys.path.insert(0, str(BAYES_DIR))

print("repository:", REPO)
print("modules   :", BAYES_DIR)
print("runtime   :", "Colab" if IS_COLAB else "local")


### 1.2 Install the locked environment

On a fresh Colab runtime this cell can deliberately restart the process once. After reconnection,
choose **Runtime > Run all** again. This is expected and prevents mixed NumPy/SciPy installations.


In [ ]:
import json
import shutil
import sysconfig
import tempfile
import time

RESTART_STATE_PATH = Path(tempfile.gettempdir()) / "patchaliasing_A_freq_env_state.json"
MAX_RESTARTS = 2


def uv_executable() -> str:
    found = shutil.which("uv")
    if found:
        return found
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])
    candidates = [
        Path(sysconfig.get_path("scripts")) / ("uv.exe" if os.name == "nt" else "uv"),
        Path(sys.executable).parent / ("uv.exe" if os.name == "nt" else "uv"),
    ]
    for candidate in candidates:
        if candidate.is_file():
            return str(candidate)
    raise FileNotFoundError("uv was installed but its executable was not found")


def load_restart_state() -> dict:
    if RESTART_STATE_PATH.is_file():
        try:
            return json.loads(RESTART_STATE_PATH.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            pass
    return {"restarts": 0}


def save_restart_state(state: dict) -> None:
    RESTART_STATE_PATH.write_text(json.dumps(state), encoding="utf-8")


def clean_imports_are_healthy() -> bool:
    probe = subprocess.run(
        [
            sys.executable,
            "-c",
            "import numpy, scipy, pandas, pyarrow, arviz, pymc, nutpie, h5netcdf",
        ],
        capture_output=True,
        text=True,
    )
    if probe.returncode != 0:
        print(probe.stderr[-2500:])
    return probe.returncode == 0


def restart_colab(reason: str) -> None:
    print("=" * 78)
    print(f"RESTARTING THE COLAB RUNTIME: {reason}")
    print("This is deliberate. After reconnection choose Runtime > Run all again.")
    print("=" * 78)
    sys.stdout.flush()
    time.sleep(2)
    os.kill(os.getpid(), 9)


if not IS_COLAB:
    print("Local runtime: dependency installation skipped; using the active environment.")
else:
    state = load_restart_state()
    UV = uv_executable()

    if state["restarts"] == 0:
        with tempfile.TemporaryDirectory() as temporary:
            requirements = Path(temporary) / "requirements.locked.txt"
            subprocess.check_call(
                [
                    UV,
                    "export",
                    "--frozen",
                    "--no-dev",
                    "--no-emit-project",
                    "--no-hashes",
                    "--output-file",
                    str(requirements),
                ],
                cwd=REPO,
            )
            subprocess.check_call(
                [UV, "pip", "install", "--python", sys.executable, "--requirement", str(requirements)]
            )

        # Required for saving the small posterior checkpoint on Python 3.12+.
        subprocess.check_call(
            [UV, "pip", "install", "--python", sys.executable, "h5netcdf", "h5py"]
        )

        # Colab's preinstalled vision wheels can be ABI-incompatible with the locked torch.
        # They are not used by this analysis.
        subprocess.run(
            [sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"],
            check=False,
            capture_output=True,
        )
        save_restart_state({"restarts": 1})
        restart_colab("locked environment installed")

    elif state["restarts"] == 1:
        if clean_imports_are_healthy():
            save_restart_state({"restarts": "verified"})
            print("Locked environment verified in the restarted runtime.")
        else:
            subprocess.check_call(
                [
                    UV,
                    "pip",
                    "install",
                    "--python",
                    sys.executable,
                    "--reinstall-package",
                    "numpy",
                    "--reinstall-package",
                    "scipy",
                ]
            )
            save_restart_state({"restarts": 2})
            restart_colab("NumPy/SciPy clean reinstall")
    else:
        if not clean_imports_are_healthy():
            raise RuntimeError(
                "The scientific imports are still broken after two restarts. Choose Runtime > "
                "Disconnect and delete runtime, reconnect to a fresh VM, and run all again."
            )
        save_restart_state({"restarts": "verified"})
        print("Locked environment verified.")


### 1.3 Choose PILOT or FULL

PILOT uses three backgrounds per generator and 1,000 retained draws. FULL requires a matching
pilot PASS, uses 100 backgrounds per generator and 2,000 retained draws, and runs recovery,
PPC and sensitivity. Pair reuse searches only the selected mode, so FULL cannot load pilot data.


In [ ]:
from __future__ import annotations

import gc
import importlib.util
import json
import math
import os
import platform
import random
import subprocess
import time
from importlib import metadata as importlib_metadata
from pathlib import Path

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import xarray as xr
from IPython.display import display
from scipy.signal.windows import hann

import bayesian_checks as bc
import checkpointing as cp
import model_loader as ml
import probe_lib as pl

RUN_MODE = "PILOT"  # Change only to "FULL" after the pilot route prints PASS.

NOTEBOOK_VERSION = "model-A-freq-standalone-v3-fixed1"
MODEL_VERSION = "A-freq-quality-difference-noncentered-v3"
RUN_ID = "model_A_freq_snr125_v3_fixed1"

QUALITY_DIFF_ATTENUATION = -0.20
QUALITY_DIFF_ROPE = 0.10


SEED = 42

TONE_SNR = 1.25
DELTA_HZ = 1.0
NFFT = 4096
BAND_HZ = (2.0, 250.0)
CONTROL_GUARD_HZ = 2.0
PRIMARY_WINDOW = "hann"
SENSITIVITY_WINDOW = "rectangular"

CORE_DESIGN_FINGERPRINT = cp.fingerprint(
    {
        "notebook_version": NOTEBOOK_VERSION,
        "model_version": MODEL_VERSION,
        "tone_snr": TONE_SNR,
        "delta_hz": DELTA_HZ,
        "nfft": NFFT,
        "band_hz": BAND_HZ,
        "control_guard_hz": CONTROL_GUARD_HZ,
        "primary_window": PRIMARY_WINDOW,
        "sensitivity_window": SENSITIVITY_WINDOW,
        "forecast_response": "forecast_with_tone_minus_forecast_background_only",
        "rounding": None,
        "response": "q_lock_minus_q_control",
        "likelihood": "StudentT",
        "nu": 4,
        "prior_scale": 0.5,
        "quality_diff_attenuation": QUALITY_DIFF_ATTENUATION,
        "quality_diff_rope": QUALITY_DIFF_ROPE,
    }
)

TRUTH_COVERAGE_MIN = 0.90
TRUTH_LOCK_CONTROL_GAP_MAX = 0.05
PPC_MIN_COVERAGE = 0.90
SENSITIVITY_MAX_SPREAD = 0.10
POSTERIOR_CUTOFF = 0.95

if RUN_MODE not in {"PILOT", "FULL"}:
    raise ValueError("RUN_MODE must be exactly 'PILOT' or 'FULL'")

IS_FULL = RUN_MODE == "FULL"
N_BG = 100 if IS_FULL else 3
DRAWS = 2000 if IS_FULL else 1000
TUNE = 2000 if IS_FULL else 1000
CHAINS = 4
TARGET_ACCEPT = 0.95
PILOT_ESS_MIN = 400
CORES = max(1, min(4, CHAINS, os.cpu_count() or 1))
BATCH_SIZE = 64
SPECTRAL_BATCH_SIZE = 512
PPC_DRAWS = 400
RECOVERY_DRAWS = 2000
RECOVERY_TUNE = 1500
PRIOR_SCALE = 0.5
PRIOR_SCALES = (0.25, 0.5, 1.0)
NU = 4
NUTS_BACKEND = "nutpie"

if NUTS_BACKEND == "nutpie" and importlib.util.find_spec("nutpie") is None:
    print("nutpie unavailable; falling back to PyMC's built-in NUTS sampler")
    NUTS_BACKEND = "pymc"

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

if IS_COLAB:
    from google.colab import drive

    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    default_drive_root = Path("/content/drive/MyDrive/patchAliasing")
else:
    default_drive_root = BAYES_DIR / "_run"

DRIVE_ROOT = Path(os.environ.get("A_FREQ_DRIVE_ROOT", str(default_drive_root)))
MODE_FOLDER = "full" if IS_FULL else "pilots"
# Optional explicit source; AUTO searches only compatible historical runs in this mode.
PAIR_REUSE = "AUTO"  # Set to "OFF" to collect new forecasts.
SOURCE_PAIRS_OVERRIDE = os.environ.get("A_FREQ_SOURCE_PAIRS", "")
OUTPUT_ROOT = DRIVE_ROOT / MODE_FOLDER / RUN_ID
PILOT_ROOT = DRIVE_ROOT / "pilots" / RUN_ID
PILOT_RESULT_PATH = PILOT_ROOT / "A_freq_final_verdict.json"
DATA_ROOT = OUTPUT_ROOT / "data"
RAW_ROOT = DATA_ROOT / "raw"
BACKGROUND_ROOT = DATA_ROOT / "backgrounds"
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints"
FIGURE_ROOT = OUTPUT_ROOT / "figures"
for directory in (OUTPUT_ROOT, DATA_ROOT, RAW_ROOT, BACKGROUND_ROOT, CHECKPOINT_ROOT, FIGURE_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

if IS_FULL:
    if not PILOT_RESULT_PATH.is_file():
        raise FileNotFoundError(
            "FULL is locked until the same notebook has completed in PILOT mode. "
            f"Missing: {PILOT_RESULT_PATH}"
        )
    pilot_result = json.loads(PILOT_RESULT_PATH.read_text(encoding="utf-8"))
    if pilot_result.get("core_design_fingerprint") != CORE_DESIGN_FINGERPRINT:
        raise ValueError("the completed pilot used a different scientific definition")
    if pilot_result.get("pilot_route_ok") is not True:
        raise ValueError("the completed pilot did not pass its estimator and convergence route")
    print("matching PILOT PASS verified:", PILOT_RESULT_PATH)

style = next(
    (name for name in ("arviz-whitegrid", "seaborn-v0_8-whitegrid") if name in plt.style.available),
    "default",
)
plt.style.use(style)

print("mode/notebook/model:", RUN_MODE, NOTEBOOK_VERSION, MODEL_VERSION)
print("outputs            :", OUTPUT_ROOT)
print("tone SNR / delta   :", TONE_SNR, DELTA_HZ, "Hz")
print("FFT/window         :", NFFT, PRIMARY_WINDOW, "+", SENSITIVITY_WINDOW)
print("backgrounds/gen    :", N_BG)
print("draws/tune/chains  :", DRAWS, TUNE, CHAINS)
print("backend/cores      :", NUTS_BACKEND, CORES)
print("REPORTABLE MODE    :", IS_FULL)
print("core design hash    :", CORE_DESIGN_FINGERPRINT)


## 2. Frequency design and estimator preflight

### 2.1 Define the paired lock/control design

TSMixup's frequency pool is used as a reproducible catalogue, not as the hidden source of the
target tone. For each geometry, controls are selected from that pool and must be at least 2 Hz
from every lock of that geometry. The nearest valid control on each side is stored. Each replicate
uses only one side, alternating deterministically, so the collected rows are exactly 50 percent
lock and 50 percent no-lock without returning to the old three-point contrast.


In [ ]:
MODELS = list(pl.DELIVERABLE3_MODELS)
GENERATORS = tuple(pl.GENERATORS)
FREQUENCY_POOL = np.asarray(pl.tsmixup_pool(), dtype=float)


def matched_control_table(P: int, S: int) -> pd.DataFrame:
    locks = np.asarray(pl.f_lock(P, S, fmin=BAND_HZ[0], fmax=BAND_HZ[1]), dtype=float)
    if not len(locks):
        raise ValueError(f"p{P}-s{S} has no in-band lock frequencies")
    distance_to_any_lock = np.min(
        np.abs(FREQUENCY_POOL[:, None] - locks[None, :]), axis=1
    )
    eligible = FREQUENCY_POOL[distance_to_any_lock >= CONTROL_GUARD_HZ - 1e-12]
    rows = []
    for site_index, f_lock in enumerate(locks):
        lower = eligible[eligible < f_lock]
        upper = eligible[eligible > f_lock]
        if not len(lower) or not len(upper):
            raise ValueError(f"p{P}-s{S} at {f_lock:g} Hz has no two-sided clean control")
        rows.append(
            {
                "model": pl.model_tag(P, S),
                "P": int(P),
                "S": int(S),
                "overlap": float((P - S) / P),
                "site_index": int(site_index),
                "site_id": f"{pl.model_tag(P, S)}@{f_lock:.9f}",
                "f_lock": float(f_lock),
                "f_control_lo": float(lower[-1]),
                "f_control_hi": float(upper[0]),
                "distance_lo": float(f_lock - lower[-1]),
                "distance_hi": float(upper[0] - f_lock),
            }
        )
    return pd.DataFrame(rows)


design = pd.concat(
    [matched_control_table(P, S) for P, S in MODELS], ignore_index=True
).sort_values(["model", "f_lock"]).reset_index(drop=True)

if design["site_id"].duplicated().any():
    raise ValueError("frequency design contains duplicate site IDs")
if len(design) != sum(len(pl.f_lock(P, S)) for P, S in MODELS):
    raise ValueError("frequency design does not cover every lock site")
if design[["distance_lo", "distance_hi"]].min().min() < CONTROL_GUARD_HZ - 1e-9:
    raise ValueError("at least one no-lock control violates CONTROL_GUARD_HZ")

print("models             :", len(MODELS))
print("lock sites         :", len(design))
print("TSMixup pool       :", len(FREQUENCY_POOL), "frequencies")
print("control distances  :", design[["distance_lo", "distance_hi"]].stack().describe())
display(design.head(12))


### 2.2 Define and unit-test the frequency estimator

DFT means discrete Fourier transform: it decomposes the 64 forecast values into candidate
frequencies. Zero-padding changes the displayed frequency grid from 8 Hz to 0.125 Hz but does not
create new signal information. The plain argmax is searched over the complete registered band,
so it is not forced to remain near the injected tone.

The quality is `q = exp(-log(2) * (error / delta)^2)`. It equals 1 at an exact match, 0.5 at
exactly 1 Hz error, and decreases smoothly beyond the tolerance.


In [ ]:
WINDOWS = {
    "hann": hann(pl.PRED, sym=False).astype(float),
    "rectangular": np.ones(pl.PRED, dtype=float),
}
FFT_FREQUENCIES = np.fft.rfftfreq(NFFT, d=1.0 / pl.FS)
FFT_BAND_MASK = (FFT_FREQUENCIES >= BAND_HZ[0]) & (FFT_FREQUENCIES <= BAND_HZ[1])
FFT_BAND_FREQUENCIES = FFT_FREQUENCIES[FFT_BAND_MASK]


def peak_metrics(values: np.ndarray, f_in: np.ndarray, window_name: str) -> pd.DataFrame:
    matrix = np.asarray(values, dtype=float)
    if matrix.ndim == 1:
        matrix = matrix[None, :]
    targets = np.asarray(f_in, dtype=float).reshape(-1)
    if matrix.shape != (len(targets), pl.PRED):
        raise ValueError(
            f"expected [n, {pl.PRED}] values and n target frequencies; got {matrix.shape}"
        )
    if window_name not in WINDOWS:
        raise ValueError(f"unknown window {window_name!r}")
    if not np.isfinite(matrix).all() or not np.isfinite(targets).all():
        raise ValueError("frequency estimator received NaN or infinite values")

    rows = []
    for start in range(0, len(matrix), SPECTRAL_BATCH_SIZE):
        stop = min(start + SPECTRAL_BATCH_SIZE, len(matrix))
        block = matrix[start:stop]
        block = block - block.mean(axis=1, keepdims=True)
        spectrum = np.abs(
            np.fft.rfft(block * WINDOWS[window_name][None, :], n=NFFT, axis=1)
        )[:, FFT_BAND_MASK]
        peak_index = np.argmax(spectrum, axis=1)
        f_prime = FFT_BAND_FREQUENCIES[peak_index]
        error = np.abs(f_prime - targets[start:stop])
        log_quality = -np.log(2.0) * np.square(error / DELTA_HZ)
        quality = np.exp(np.maximum(log_quality, np.log(np.finfo(float).tiny)))
        band_median = np.median(spectrum, axis=1)
        prominence_ratio = spectrum[np.arange(len(spectrum)), peak_index] / np.maximum(
            band_median, np.finfo(float).tiny
        )
        rows.append(
            pd.DataFrame(
                {
                    "f_prime": f_prime,
                    "frequency_error": error,
                    "log_quality": log_quality,
                    "quality": quality,
                    "contrast_d": 2.0 * quality - 1.0,
                    "inside_delta": error <= DELTA_HZ + 1e-12,
                    "peak_prominence_ratio": prominence_ratio,
                }
            )
        )
    return pd.concat(rows, ignore_index=True)



# Pure-tone unit checks, not scientific results.
t = np.arange(pl.PRED) / pl.FS
test_frequencies = np.array([25.0, 32.0, 35.25])
test_signals = np.stack([np.sin(2 * np.pi * f * t + 0.37) for f in test_frequencies])
test_metrics = peak_metrics(test_signals, test_frequencies, PRIMARY_WINDOW)
if float(test_metrics["frequency_error"].max()) > 0.125 + 1e-12:
    raise AssertionError("frequency estimator failed its isolated-tone grid check")

anchor_errors = np.array([0.0, 1.0, 2.0, 3.0])
anchor_q = np.exp(-np.log(2.0) * np.square(anchor_errors / DELTA_HZ))
anchor = pd.DataFrame(
    {"error_hz": anchor_errors, "q": anchor_q, "d": 2.0 * anchor_q - 1.0}
)
print("native DFT spacing :", pl.FS / pl.PRED, "Hz")
print("zero-padded spacing:", pl.FS / NFFT, "Hz")
display(test_metrics)
display(anchor)

grid = np.linspace(0, 4, 500)
curve = np.exp(-np.log(2.0) * np.square(grid / DELTA_HZ))
figure, axis = plt.subplots(figsize=(7, 4))
axis.plot(grid, curve, label="q(error)")
axis.axvline(DELTA_HZ, color="black", ls="--", lw=1, label="delta = 1 Hz")
axis.set(xlabel="absolute frequency error [Hz]", ylabel="localisation quality q", ylim=(-0.02, 1.02))
axis.legend()
figure.tight_layout()
plt.show()


### 2.3 Create an immutable run manifest

The manifest fingerprints the exact design, repository revision, helper code, model identities,
sampling settings and output namespace. A mismatch stops rather than silently combining artifacts
from different definitions.


In [ ]:
MANIFEST_PATH = OUTPUT_ROOT / "analysis_manifest.json"


def package_version(name: str):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return None


repository_revision = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
checkpoint_identities = {
    pl.model_tag(P, S): ml.checkpoint_identity(P, S) for P, S in MODELS
}

analysis_spec = {
    "schema_version": 1,
    "notebook_version": NOTEBOOK_VERSION,
    "model_version": MODEL_VERSION,
    "run_mode": RUN_MODE,
    "run_id": RUN_ID,
    "repository_revision": repository_revision,
    "helper_sha256": {
        name: cp.sha256_file(BAYES_DIR / name)
        for name in ("probe_lib.py", "bayesian_checks.py", "checkpointing.py", "model_loader.py")
    },
    "checkpoint_identities": checkpoint_identities,
    "models": [pl.model_tag(P, S) for P, S in MODELS],
    "generators": list(GENERATORS),
    "design": {
        "tone_snr_amplitude_ratio": TONE_SNR,
        "delta_hz": DELTA_HZ,
        "nfft": NFFT,
        "band_hz": list(BAND_HZ),
        "control_guard_hz": CONTROL_GUARD_HZ,
        "primary_window": PRIMARY_WINDOW,
        "sensitivity_window": SENSITIVITY_WINDOW,
        "forecast_response": "forecast_with_tone_minus_forecast_background_only",
        "rounding": None,
        "n_backgrounds_per_generator": N_BG,
        "one_phase_per_background_site": True,
        "n_sites": int(len(design)),
        "frequency_pool_sha256": cp.fingerprint(FREQUENCY_POOL.tolist()),
        "frequency_design_sha256": cp.fingerprint(design.to_dict(orient="records")),
    },
    "statistical_model": {
        "response": "q_lock_minus_q_control", "likelihood": "StudentT", "nu": NU,
        "noncentred_zero_sum": True, "prior_scale": PRIOR_SCALE,
        "prior_scales": list(PRIOR_SCALES), "ppc_draws": PPC_DRAWS,
        "recovery_draws": RECOVERY_DRAWS, "recovery_tune": RECOVERY_TUNE,
        "seed": SEED,
    },
    "sampling": {
        "draws": DRAWS,
        "tune": TUNE,
        "chains": CHAINS,
        "cores": CORES,
        "target_accept": TARGET_ACCEPT,
        "backend": NUTS_BACKEND,
        "log_likelihood": False,
    },
    "thresholds": {
        "truth_coverage_min": TRUTH_COVERAGE_MIN,
        "truth_lock_control_gap_max": TRUTH_LOCK_CONTROL_GAP_MAX,
        "rhat_max": bc.RHAT_MAX,
        "ess_min_full": bc.ESS_MIN,
        "ess_min_pilot": PILOT_ESS_MIN,
        "posterior_cutoff": POSTERIOR_CUTOFF,
        "quality_diff_attenuation": QUALITY_DIFF_ATTENUATION,
        "quality_diff_rope": QUALITY_DIFF_ROPE,
        "ppc_min_coverage": PPC_MIN_COVERAGE,
        "sensitivity_max_spread": SENSITIVITY_MAX_SPREAD,
    },
    "packages": {
        name: package_version(name)
        for name in ("numpy", "scipy", "pandas", "pyarrow", "pymc", "arviz", "nutpie", "torch")
    },
    "python": platform.python_version(),
}

def resolve_pair_source(current_spec: dict):
    if PAIR_REUSE not in {"AUTO", "OFF"}:
        raise ValueError("PAIR_REUSE must be AUTO or OFF")
    if PAIR_REUSE == "OFF":
        return None, None
    candidates = ([Path(SOURCE_PAIRS_OVERRIDE)] if SOURCE_PAIRS_OVERRIDE else [
        DRIVE_ROOT / MODE_FOLDER / run / "data" / "A_freq_pairs.parquet"
        for run in ("model_A_freq_snr125_v3", "model_A_freq_snr125_v2", "model_A_freq_snr125_v1")
    ])
    for path in candidates:
        if not path.is_file():
            if SOURCE_PAIRS_OVERRIDE:
                raise FileNotFoundError(path)
            continue
        source_manifest_path = path.parent.parent / "analysis_manifest.json"
        source_manifest = json.loads(source_manifest_path.read_text(encoding="utf-8"))
        source_spec = source_manifest["analysis_spec"]
        if source_manifest.get("analysis_fingerprint") != cp.fingerprint(source_spec):
            raise ValueError(f"source manifest fingerprint mismatch: {source_manifest_path}")
        # Statistical versions differ; the forecast/measurement specification must match.
        for field in ("run_mode", "models", "generators", "checkpoint_identities", "design"):
            if source_spec.get(field) != current_spec[field]:
                raise ValueError(f"incompatible source {field}: {path}; choose a compatible source or PAIR_REUSE='OFF'")
        for helper in ("probe_lib.py", "model_loader.py"):
            if source_spec.get("helper_sha256", {}).get(helper) != current_spec["helper_sha256"][helper]:
                raise ValueError(f"source collection helper changed: {helper}")
        relative = path.relative_to(path.parent.parent).as_posix()
        expected_hash = source_manifest.get("artifacts", {}).get(relative, {}).get("sha256")
        actual_hash = cp.sha256_file(path)
        if not expected_hash or actual_hash != expected_hash:
            raise ValueError(f"source pair-table hash mismatch: {path}")
        return path, {
            "path": str(path), "sha256": actual_hash,
            "source_analysis_fingerprint": source_manifest["analysis_fingerprint"],
            "source_manifest_sha256": cp.sha256_file(source_manifest_path),
        }
    return None, None


SOURCE_PAIRS_PATH, SOURCE_PAIR_PROVENANCE = resolve_pair_source(analysis_spec)
analysis_spec["pair_source"] = SOURCE_PAIR_PROVENANCE
print("pair source:", SOURCE_PAIRS_PATH or "new Chronos collection")

ANALYSIS_FINGERPRINT = cp.fingerprint(analysis_spec)

if MANIFEST_PATH.is_file():
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    if manifest.get("analysis_fingerprint") != ANALYSIS_FINGERPRINT:
        raise ValueError(
            "Existing output manifest belongs to a different design. Increment RUN_ID rather "
            "than overwriting or mixing results."
        )
else:
    manifest = {
        "schema_version": 1,
        "analysis_fingerprint": ANALYSIS_FINGERPRINT,
        "analysis_spec": analysis_spec,
        "artifacts": {},
    }
    cp.atomic_json(MANIFEST_PATH, manifest)


def refresh_manifest() -> dict:
    current = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    if current.get("analysis_fingerprint") != ANALYSIS_FINGERPRINT:
        raise ValueError("analysis manifest changed during this runtime")
    return current


def record_artifact(path: Path) -> None:
    current = refresh_manifest()
    relative = str(path.relative_to(OUTPUT_ROOT)).replace("\\", "/")
    current.setdefault("artifacts", {})[relative] = {
        "sha256": cp.sha256_file(path),
        "bytes": int(path.stat().st_size),
    }
    cp.atomic_json(MANIFEST_PATH, current)


def artifact_is_valid(path: Path) -> bool:
    current = refresh_manifest()
    relative = str(path.relative_to(OUTPUT_ROOT)).replace("\\", "/")
    entry = current.get("artifacts", {}).get(relative)
    if not path.is_file():
        if entry is not None:
            raise ValueError(f"manifest records a missing artifact: {relative}")
        return False
    if entry is None:
        raise ValueError(f"untracked artifact exists: {relative}")
    if cp.sha256_file(path) != entry.get("sha256"):
        raise ValueError(f"artifact hash mismatch: {relative}")
    return True


print("repository revision :", repository_revision)
print("analysis fingerprint:", ANALYSIS_FINGERPRINT)
print("manifest            :", MANIFEST_PATH)


## 3. Collect A-freq observations

### 3.1 Generate or reuse the canonical backgrounds

Every background is exactly 544 samples, centred and scaled to standard deviation 1 by the project
helper. Saving them separately makes a disconnected Colab run resumable without regenerating
KernelSynth. These arrays contain no injected target tone.

When a verified pair table is available, this entire generation step is skipped.


In [ ]:
if SOURCE_PAIRS_PATH is not None:
    print("Verified pair reuse: background generation skipped.")
else:
    BACKGROUND_INDEX_PATH = DATA_ROOT / "background_index.parquet"


    def atomic_npy(path: Path, values: np.ndarray) -> None:
        temporary = path.with_name(path.stem + ".tmp.npy")
        np.save(temporary, np.asarray(values, dtype=np.float32))
        os.replace(temporary, path)


    background_rows = []
    background_started = time.time()
    background_total = len(GENERATORS) * N_BG
    background_done = 0
    for generator in GENERATORS:
        for bg_id in range(N_BG):
            path = BACKGROUND_ROOT / f"{generator}__bg{bg_id:03d}.npy"
            if path.is_file():
                values = np.load(path)
                state = "reused"
            else:
                values = pl.background(generator, pl.CANON_LEN, seed=10_000 + bg_id)
                atomic_npy(path, values)
                state = "saved"
            values = np.asarray(values, dtype=float)
            if values.shape != (pl.CANON_LEN,):
                raise ValueError(f"invalid background shape: {path} -> {values.shape}")
            if not np.isfinite(values).all():
                raise ValueError(f"non-finite background: {path}")
            mean = float(values.mean())
            sd = float(values.std())
            if abs(mean) > 1e-5 or abs(sd - 1.0) > 1e-5:
                raise ValueError(f"background is not canonical mean-0/std-1: {path}")
            background_rows.append(
                {
                    "generator": generator,
                    "bg_id": bg_id,
                    "path": str(path),
                    "sha256": cp.sha256_file(path),
                    "mean": mean,
                    "std": sd,
                }
            )
            background_done += 1
            if background_done == 1 or background_done % 10 == 0 or background_done == background_total:
                elapsed = time.time() - background_started
                eta = elapsed / background_done * (background_total - background_done)
                print(
                    f"background [{background_done}/{background_total}] {generator} {bg_id}: "
                    f"{state}; elapsed {elapsed/60:.1f} min; ETA {eta/60:.1f} min"
                )

    background_index = pd.DataFrame(background_rows)
    cp.atomic_parquet(BACKGROUND_INDEX_PATH, background_index)
    record_artifact(BACKGROUND_INDEX_PATH)
    BACKGROUND_SET_SHA256 = cp.fingerprint(
        background_index.sort_values(["generator", "bg_id"])[
            ["generator", "bg_id", "sha256"]
        ].to_dict(orient="records")
    )
    display(
        background_index.groupby("generator").agg(
            n=("bg_id", "size"), max_abs_mean=("mean", lambda x: np.max(np.abs(x))),
            max_std_error=("std", lambda x: np.max(np.abs(x - 1.0))),
        )
    )
    print("background set hash:", BACKGROUND_SET_SHA256)


### 3.2 Forecast each geometry, one resumable shard at a time

This is the only GPU-heavy collection cell. For each model and background it first forecasts the
background alone. Those baseline forecasts are reused for every tone on that background. Frequency
estimation is then applied to the incremental response `forecast_with_tone - forecast_background`.
Models are loaded sequentially and released before the next geometry. Progress reports show
completed geometries, elapsed time and ETA. An existing shard is reused only when its sidecar
records the same run fingerprint, checkpoint identity and file hash.


In [ ]:
def background_array(generator: str, bg_id: int) -> np.ndarray:
    row = background_index[
        background_index["generator"].eq(generator)
        & background_index["bg_id"].eq(bg_id)
    ]
    if len(row) != 1:
        raise ValueError(f"background index mismatch for {generator} #{bg_id}")
    path = Path(row.iloc[0]["path"])
    if cp.sha256_file(path) != row.iloc[0]["sha256"]:
        raise ValueError(f"background hash mismatch: {path}")
    return np.asarray(np.load(path), dtype=np.float32)


def choose_phase(f_lock: float, site_index: int, bg_id: int) -> tuple[int, float]:
    options = np.asarray(pl.phases_Sf(f_lock, n_max=10), dtype=float)
    phase_index = int((bg_id + 3 * site_index) % len(options))
    return phase_index, float(options[phase_index])


def add_metrics(frame: pd.DataFrame, values: np.ndarray, prefix: str, window_name: str) -> None:
    metrics = peak_metrics(values, frame["f_in"].to_numpy(float), window_name)
    for column in metrics.columns:
        frame[f"{prefix}_{window_name}_{column}"] = metrics[column].to_numpy()


def collect_geometry(P: int, S: int) -> pd.DataFrame:
    model = pl.model_tag(P, S)
    shard_path = RAW_ROOT / f"A_freq__{model}.parquet"
    sidecar_path = RAW_ROOT / f"A_freq__{model}.json"
    expected_identity = checkpoint_identities[model]

    if shard_path.is_file() or sidecar_path.is_file():
        if not (shard_path.is_file() and sidecar_path.is_file()):
            raise ValueError(f"partial shard pair exists for {model}; use a new RUN_ID")
        sidecar = json.loads(sidecar_path.read_text(encoding="utf-8"))
        if sidecar.get("analysis_fingerprint") != ANALYSIS_FINGERPRINT:
            raise ValueError(f"stale design fingerprint for {model}")
        if sidecar.get("checkpoint_identity") != expected_identity:
            raise ValueError(f"checkpoint identity changed for {model}")
        if sidecar.get("background_set_sha256") != BACKGROUND_SET_SHA256:
            raise ValueError(f"background set changed for {model}")
        if sidecar.get("sha256") != cp.sha256_file(shard_path):
            raise ValueError(f"shard hash mismatch for {model}")
        print("reusing completed shard:", model)
        return pd.read_parquet(shard_path)

    site_table = design[design["model"].eq(model)].sort_values("site_index")
    probe = pl.Probe(P, S, batch_size=BATCH_SIZE)
    if probe.checkpoint_identity != expected_identity:
        probe.close()
        raise ValueError(f"loaded checkpoint identity differs for {model}")

    geometry_pairs = []
    try:
        for generator in GENERATORS:
            contexts = []
            futures = []
            background_futures = []
            metadata = []
            generator_backgrounds = [
                background_array(generator, bg_id) for bg_id in range(N_BG)
            ]
            baseline_contexts = np.stack(
                [background[: pl.CTX] for background in generator_backgrounds]
            )
            baseline_predictions = probe.forecast(baseline_contexts)
            for site in site_table.itertuples(index=False):
                for bg_id in range(N_BG):
                    background = generator_backgrounds[bg_id]
                    phase_index, phase = choose_phase(site.f_lock, site.site_index, bg_id)
                    use_lower = ((bg_id + site.site_index) % 2) == 0
                    f_control = site.f_control_lo if use_lower else site.f_control_hi
                    control_side = "lo" if use_lower else "hi"
                    for role, f_in in (("lock", site.f_lock), ("control", f_control)):
                        full = (
                            background
                            + pl.make_tone(float(f_in), phase, pl.CANON_LEN, amp=TONE_SNR)
                        ).astype(np.float32)
                        contexts.append(full[: pl.CTX])
                        futures.append(full[pl.CTX : pl.CTX + pl.PRED])
                        background_futures.append(
                            background[pl.CTX : pl.CTX + pl.PRED]
                        )
                        metadata.append(
                            {
                                "model": model,
                                "P": P,
                                "S": S,
                                "overlap": float((P - S) / P),
                                "generator": generator,
                                "bg_id": bg_id,
                                "site_index": int(site.site_index),
                                "site_id": site.site_id,
                                "f_lock": float(site.f_lock),
                                "f_control": float(f_control),
                                "control_side": control_side,
                                "phase_index": phase_index,
                                "phase": phase,
                                "role": role,
                                "f_in": float(f_in),
                            }
                        )

            long = pd.DataFrame(metadata)
            context_matrix = np.stack(contexts)
            future_matrix = np.stack(futures)
            background_future_matrix = np.stack(background_futures)
            started = time.time()
            predictions = probe.forecast(context_matrix)
            print(
                f"{model} {generator}: {len(long):,} forecasts in "
                f"{(time.time() - started)/60:.1f} min"
            )
            if predictions.shape != future_matrix.shape:
                raise ValueError(
                    f"{model} forecast shape {predictions.shape} != truth shape {future_matrix.shape}"
                )

            baseline_for_rows = baseline_predictions[long["bg_id"].to_numpy(int)]
            predicted_response = predictions - baseline_for_rows
            true_response = future_matrix - background_future_matrix
            long["pred_response_rms"] = np.sqrt(np.mean(np.square(predicted_response), axis=1))
            long["truth_response_rms"] = np.sqrt(np.mean(np.square(true_response), axis=1))

            for window_name in (PRIMARY_WINDOW, SENSITIVITY_WINDOW):
                add_metrics(long, predicted_response, "pred", window_name)
                add_metrics(long, true_response, "truth", window_name)

            key = [
                "model", "P", "S", "overlap", "generator", "bg_id", "site_index",
                "site_id", "f_lock", "f_control", "control_side", "phase_index", "phase",
            ]
            value_columns = [
                column for column in long.columns
                if column not in key + ["role", "f_in"]
            ]
            wide_parts = []
            for role in ("lock", "control"):
                part = long[long["role"].eq(role)][key + value_columns].copy()
                part = part.rename(columns={column: f"{column}_{role}" for column in value_columns})
                wide_parts.append(part)
            wide = wide_parts[0].merge(wide_parts[1], on=key, how="inner", validate="one_to_one")

            for prefix in ("pred_hann", "pred_rectangular", "truth_hann", "truth_rectangular"):
                wide[f"{prefix}_d_quality"] = (
                    wide[f"{prefix}_quality_lock"].to_numpy(float)
                    - wide[f"{prefix}_quality_control"].to_numpy(float)
                )

            geometry_pairs.append(wide)
            del (
                contexts, futures, background_futures, metadata, generator_backgrounds,
                baseline_contexts, baseline_predictions, context_matrix, future_matrix,
                background_future_matrix, predictions, baseline_for_rows, predicted_response,
                true_response, long, wide,
            )
            gc.collect()
    finally:
        probe.close()
        del probe
        gc.collect()

    result = pd.concat(geometry_pairs, ignore_index=True)
    expected_rows = len(site_table) * N_BG * len(GENERATORS)
    if len(result) != expected_rows:
        raise ValueError(f"{model}: expected {expected_rows} pairs, got {len(result)}")
    cp.atomic_parquet(shard_path, result)
    sidecar = {
        "analysis_fingerprint": ANALYSIS_FINGERPRINT,
        "checkpoint_identity": expected_identity,
        "background_set_sha256": BACKGROUND_SET_SHA256,
        "rows": int(len(result)),
        "sha256": cp.sha256_file(shard_path),
    }
    cp.atomic_json(sidecar_path, sidecar)
    print("saved shard:", shard_path)
    return result


def add_quality_difference_columns(frame):
    frame = frame.copy()
    for prefix in ("pred_hann", "pred_rectangular",
                   "truth_hann", "truth_rectangular"):
        frame[f"{prefix}_d_quality"] = (
            frame[f"{prefix}_quality_lock"].to_numpy(float)
            - frame[f"{prefix}_quality_control"].to_numpy(float)
        )
    return frame

if SOURCE_PAIRS_PATH is not None:
    if cp.sha256_file(SOURCE_PAIRS_PATH) != SOURCE_PAIR_PROVENANCE["sha256"]:
        raise ValueError("source pairs changed since preflight")
    observations = add_quality_difference_columns(pd.read_parquet(SOURCE_PAIRS_PATH))
    print("reused pairs:", len(observations))
else:
    collection_started = time.time()
    geometry_frames = []
    for model_index, (P, S) in enumerate(MODELS, start=1):
        started = time.time()
        geometry_frames.append(collect_geometry(P, S))
        elapsed = time.time() - collection_started
        eta = elapsed / model_index * (len(MODELS) - model_index)
        print(
            f"geometry [{model_index}/{len(MODELS)}] p{P}-s{S} complete; "
            f"cell {((time.time()-started)/60):.1f} min; elapsed {elapsed/60:.1f} min; "
            f"ETA {eta/60:.1f} min"
        )

    observations = pd.concat(geometry_frames, ignore_index=True)
    del geometry_frames


gc.collect()


### 3.3 Validate and freeze the collected table

The intended grain is one matched pair per model, generator, background and lock site. These checks
reject missing cells, duplicate pairs, contaminated controls, non-finite results, wrong background
counts and accidental frequency rounding.


In [ ]:
OBSERVATIONS_PATH = DATA_ROOT / "A_freq_pairs.parquet"
PAIR_KEY = ["model", "generator", "bg_id", "site_id"]
EXPECTED_ROWS = len(design) * len(GENERATORS) * N_BG

if len(observations) != EXPECTED_ROWS:
    raise ValueError(f"expected {EXPECTED_ROWS:,} matched pairs, got {len(observations):,}")
if observations[PAIR_KEY].duplicated().any():
    raise ValueError("duplicate matched-pair keys")
if set(observations["model"]) != {pl.model_tag(P, S) for P, S in MODELS}:
    raise ValueError("model coverage mismatch")
if set(observations["generator"]) != set(GENERATORS):
    raise ValueError("generator coverage mismatch")
if observations.groupby(["model", "generator"])["bg_id"].nunique().ne(N_BG).any():
    raise ValueError("background coverage mismatch")

metric_columns = [
    column for column in observations.columns
    if any(
        token in column
        for token in (
            "f_prime", "frequency_error", "log_quality", "quality", "pair_share", "d_freq", "response_rms"
        )
    )
]
if not np.isfinite(observations[metric_columns].to_numpy(float)).all():
    raise ValueError("non-finite frequency metrics in the collected table")

locks_by_model = {
    model: np.asarray(pl.f_lock(int(group["P"].iloc[0]), int(group["S"].iloc[0])), float)
    for model, group in observations.groupby("model", observed=True)
}
minimum_control_distance = np.inf
for model, group in observations.groupby("model", observed=True):
    distances = np.min(
        np.abs(group["f_control"].to_numpy(float)[:, None] - locks_by_model[model][None, :]),
        axis=1,
    )
    minimum_control_distance = min(minimum_control_distance, float(distances.min()))
if minimum_control_distance < CONTROL_GUARD_HZ - 1e-9:
    raise ValueError("a collected control is too close to a lock frequency")

# Check the complete pair grain and its registered geometry, not just row counts.
expected_keys = {
    (row.model, generator, bg_id, row.site_id)
    for row in design.itertuples(index=False)
    for generator in GENERATORS for bg_id in range(N_BG)
}
if set(observations[PAIR_KEY].itertuples(index=False, name=None)) != expected_keys:
    raise ValueError("pair keys differ from the registered design")
registered = observations.merge(
    design, on=["model", "site_id"], suffixes=("", "_registered"), validate="many_to_one"
)
for field in ("P", "S", "overlap", "site_index", "f_lock"):
    if not np.allclose(registered[field], registered[field + "_registered"], rtol=0, atol=1e-9):
        raise ValueError(f"source geometry mismatch: {field}")
lower = ((registered["bg_id"] + registered["site_index"]) % 2) == 0
expected_control = np.where(lower, registered["f_control_lo"], registered["f_control_hi"])
if not np.allclose(registered["f_control"], expected_control, rtol=0, atol=1e-9):
    raise ValueError("source controls differ from the registered alternating design")
for prefix in ("pred_hann", "pred_rectangular", "truth_hann", "truth_rectangular"):
    for role in ("lock", "control"):
        q = observations[f"{prefix}_quality_{role}"].to_numpy(float)
        error = observations[f"{prefix}_frequency_error_{role}"].to_numpy(float)
        if np.any((q < 0) | (q > 1)) or not np.allclose(
            q, np.exp(-np.log(2) * (error / DELTA_HZ)**2), rtol=1e-10, atol=1e-14
        ):
            raise ValueError(f"invalid quality definition: {prefix}/{role}")
    delta = observations[f"{prefix}_quality_lock"] - observations[f"{prefix}_quality_control"]
    if not np.allclose(observations[f"{prefix}_d_quality"], delta, rtol=0, atol=1e-14):
        raise ValueError(f"incorrect quality difference: {prefix}")

cp.atomic_parquet(OBSERVATIONS_PATH, observations)
record_artifact(OBSERVATIONS_PATH)
print("data-quality checks: PASS")
print("matched pairs       :", len(observations))
print(
    "Chronos forecasts    :",
    2 * len(observations) + len(MODELS) * len(GENERATORS) * N_BG,
    "(paired tones plus reusable background-only baselines)",
)
print("minimum ctrl distance:", minimum_control_distance, "Hz")
print("saved               :", OBSERVATIONS_PATH)
display(
    observations.groupby(["model", "generator"], observed=True)
    .size().rename("pairs").unstack("generator")
)
display(
    observations.groupby("generator", observed=True)[
        [
            "pred_response_rms_lock", "pred_response_rms_control",
            "truth_response_rms_lock", "truth_response_rms_control",
        ]
    ].median()
)

print("Response preflight (before MCMC)")
display(observations["pred_hann_d_quality"].quantile([0, .01, .25, .5, .75, .99, 1]))
print("Fraction |d_quality| <= 0.01:", observations["pred_hann_d_quality"].abs().le(.01).mean())


## 4. Estimator validity on the true future

Before judging Chronos, the same subtraction and extractor must recover the injected frequency
from `(true background + tone) - true background`, which is exactly the planted tone over the
64-point continuation. FULL mode requires at least 90 percent within 1 Hz in every
model-by-generator-by-role cell, for both windows. It also requires no more than a 0.05 difference
between mean lock and control quality produced by the estimator itself. Failure here means the
measurement is invalid at SNR 1.25; it is not evidence against or for H1.


In [ ]:
truth_rows = []
for window_name in (PRIMARY_WINDOW, SENSITIVITY_WINDOW):
    for role in ("lock", "control"):
        grouped = observations.groupby(["model", "generator"], observed=True)
        for (model, generator), group in grouped:
            truth_rows.append(
                {
                    "window": window_name,
                    "role": role,
                    "model": model,
                    "generator": generator,
                    "n": len(group),
                    "inside_rate": float(group[f"truth_{window_name}_inside_delta_{role}"].mean()),
                    "mean_quality": float(group[f"truth_{window_name}_quality_{role}"].mean()),
                    "median_error_hz": float(group[f"truth_{window_name}_frequency_error_{role}"].median()),
                }
            )

truth_table = pd.DataFrame(truth_rows)
coverage_ok = bool((truth_table["inside_rate"] >= TRUTH_COVERAGE_MIN).all())
gap_rows = []
for window_name in (PRIMARY_WINDOW, SENSITIVITY_WINDOW):
    pivot = truth_table[truth_table["window"].eq(window_name)].pivot(
        index=["model", "generator"], columns="role", values="mean_quality"
    )
    for (model, generator), row in pivot.iterrows():
        gap_rows.append(
            {
                "window": window_name,
                "model": model,
                "generator": generator,
                "lock_minus_control": float(row["lock"] - row["control"]),
            }
        )
truth_gap_table = pd.DataFrame(gap_rows)
truth_gap_ok = bool(
    truth_gap_table["lock_minus_control"].abs().max() <= TRUTH_LOCK_CONTROL_GAP_MAX
)
TRUTH_ESTIMATOR_OK = bool(coverage_ok and truth_gap_ok)

TRUTH_TABLE_PATH = OUTPUT_ROOT / "truth_estimator_quality.parquet"
TRUTH_GAP_PATH = OUTPUT_ROOT / "truth_estimator_lock_control_gap.parquet"
cp.atomic_parquet(TRUTH_TABLE_PATH, truth_table)
cp.atomic_parquet(TRUTH_GAP_PATH, truth_gap_table)
record_artifact(TRUTH_TABLE_PATH)
record_artifact(TRUTH_GAP_PATH)

display(truth_table.sort_values("inside_rate").head(20))
display(truth_gap_table.reindex(truth_gap_table["lock_minus_control"].abs().sort_values(ascending=False).index).head(20))
print("minimum true-future coverage:", truth_table["inside_rate"].min())
print("maximum absolute lock/control extractor gap:", truth_gap_table["lock_minus_control"].abs().max())
print("TRUE-FUTURE ESTIMATOR GATE:", "PASS" if TRUTH_ESTIMATOR_OK else "FAIL")
if not TRUTH_ESTIMATOR_OK:
    print("Do not interpret any model fit. Revisit SNR, delta or the peak rule before FULL sampling.")


## 5. Bayesian Model A-freq

### 5.1 Hierarchical Student-t(4) model on paired quality differences

`d_quality = q_lock - q_control`. The population location `beta_bar` is negative when
lock localisation is worse. `delta_O` and `delta_P` describe the overlap and centred log-patch
slopes. Configuration, harmonic and background deviations use standard sum-to-zero latent
normals multiplied by their respective scale (non-centred parametrization).

Regression coefficients and positive scales retain Student-t(4)/half-Student-t(4) priors.
The likelihood also uses Student-t(4), with residual scale `sigma`. Its unbounded support
is an approximation to the bounded response; convergence and predictive checks must be assessed.


In [ ]:
def codes(series: pd.Series) -> tuple[np.ndarray, np.ndarray]:
    integer_codes, levels = pd.factorize(series, sort=False)
    return np.asarray(integer_codes, dtype=int), np.asarray(levels)


def overlap_scaled(frame: pd.DataFrame, configuration_levels: np.ndarray) -> np.ndarray:
    values = (
        frame.groupby("model", observed=True)["overlap"]
        .first().reindex(configuration_levels).to_numpy(float)
    )
    return (values - values.mean()) / 0.5


def log_patch_centred(frame: pd.DataFrame, configuration_levels: np.ndarray) -> np.ndarray:
    patch = (
        frame.groupby("model", observed=True)["P"]
        .first().reindex(configuration_levels).to_numpy(float)
    )
    values = np.log(patch)
    return values - values.mean()


def model_A_freq(
    frame: pd.DataFrame,
    response_column: str = "pred_hann_d_quality",
    prior_scale: float = PRIOR_SCALE,
    predictors: str = "both",
) -> pm.Model:
    if predictors not in {"both", "overlap", "patch", "none"}:
        raise ValueError("predictors must be both, overlap, patch or none")

    configuration_code, configuration_levels = codes(
        frame["model"].astype(str)
    )
    harmonic_code, harmonic_levels = codes(
        frame["f_lock"].map(lambda x: f"{x:.9f}")
    )
    background_code, background_levels = codes(
        frame["generator"].astype(str)
        + "#"
        + frame["bg_id"].astype(str)
    )

    overlap_t = overlap_scaled(frame, configuration_levels)
    log_patch_t = log_patch_centred(frame, configuration_levels)
    observed_contrast = frame[response_column].to_numpy(float)

    if not np.isfinite(observed_contrast).all():
        raise ValueError("non-finite frequency contrasts")

    coordinates = {
        "config": configuration_levels,
        "harmonic": harmonic_levels,
        "background": background_levels,
        "obs": np.arange(len(frame)),
    }

    with pm.Model(coords=coordinates) as model:
        beta_bar = pm.StudentT(
            "beta_bar",
            nu=NU,
            mu=0.0,
            sigma=prior_scale,
        )
        delta_O = pm.StudentT(
            "delta_O",
            nu=NU,
            mu=0.0,
            sigma=prior_scale,
        )
        delta_P = pm.StudentT(
            "delta_P",
            nu=NU,
            mu=0.0,
            sigma=prior_scale,
        )

        configuration_mean = beta_bar
        if predictors in {"both", "overlap"}:
            configuration_mean = (
                configuration_mean + delta_O * overlap_t
            )
        if predictors in {"both", "patch"}:
            configuration_mean = (
                configuration_mean + delta_P * log_patch_t
            )

        tau = pm.HalfStudentT(
            "tau",
            nu=NU,
            sigma=prior_scale,
        )
        z_config = pm.ZeroSumNormal(
            "z_config",
            sigma=1.0,
            dims="config",
        )
        beta = pm.Deterministic(
            "beta",
            configuration_mean + tau * z_config,
            dims="config",
        )

        sigma_harm = pm.HalfStudentT(
            "sigma_harm",
            nu=NU,
            sigma=prior_scale,
        )
        z_harm = pm.ZeroSumNormal(
            "z_harm",
            sigma=1.0,
            dims="harmonic",
        )
        u_harm = pm.Deterministic(
            "u_harm",
            sigma_harm * z_harm,
            dims="harmonic",
        )

        sigma_bg = pm.HalfStudentT(
            "sigma_bg",
            nu=NU,
            sigma=prior_scale,
        )
        z_bg = pm.ZeroSumNormal(
            "z_bg",
            sigma=1.0,
            dims="background",
        )
        u_bg = pm.Deterministic(
            "u_bg",
            sigma_bg * z_bg,
            dims="background",
        )

        location = (
            beta[configuration_code]
            + u_harm[harmonic_code]
            + u_bg[background_code]
        )

        sigma = pm.HalfStudentT(
            "sigma",
            nu=NU,
            sigma=prior_scale,
        )

        pm.StudentT(
            "d_quality",
            nu=NU,
            mu=location,
            sigma=sigma,
            observed=observed_contrast,
            dims="obs",
        )

        pm.Deterministic(
            "quality_difference",
            beta_bar,
        )

    return model


primary_model = model_A_freq(observations)
print("model version:", MODEL_VERSION)
print("free variables:", [variable.name for variable in primary_model.free_RVs])
print(
    "levels:",
    {
        "config": len(primary_model.coords["config"]),
        "harmonic": len(primary_model.coords["harmonic"]),
        "background": len(primary_model.coords["background"]),
        "observations": len(primary_model.coords["obs"]),
    },
)


### 5.2 Fit or resume the primary Hann model

The progress table gives the timing estimate. `CORES` adapts to the available CPUs; when fewer
cores than chains are available, chains may queue. Observation-wise log likelihood is disabled.
A new run namespace prevents loading any earlier Beta, log-ratio or incomplete-v3 posterior.


In [ ]:
def checkpoint_path(label: str) -> Path:
    return CHECKPOINT_ROOT / f"{label}.nc"


def save_checkpoint(idata: az.InferenceData, path: Path) -> None:
    cp.atomic_netcdf(path, idata)
    record_artifact(path)
    print("checkpoint saved:", path)


def fit_or_load(
    label: str,
    model: pm.Model,
    draws: int = DRAWS,
    tune: int = TUNE,
    target_accept: float = TARGET_ACCEPT,
) -> az.InferenceData:
    path = checkpoint_path(label)
    if artifact_is_valid(path):
        print("loading completed checkpoint:", path)
        return az.from_netcdf(path)
    kwargs = {}
    if NUTS_BACKEND != "pymc":
        kwargs["nuts_sampler"] = NUTS_BACKEND
    print(
        f"sampling {label}: draws={draws}, tune={tune}, chains={CHAINS}, "
        f"cores={CORES}, observations={len(model.coords['obs']):,}"
    )
    started = time.time()
    with model:
        idata = pm.sample(
            draws=draws,
            tune=tune,
            chains=CHAINS,
            cores=CORES,
            random_seed=SEED,
            target_accept=target_accept,
            progressbar=True,
            idata_kwargs={"log_likelihood": False},
            **kwargs,
        )
    print(f"sampling completed in {(time.time()-started)/60:.1f} min")
    save_checkpoint(idata, path)
    return idata


idata_primary = fit_or_load("04_A_freq_primary_hann_v3", primary_model)


### 5.3 Diagnose the primary fit and calculate H1-freq

R-hat compares variation within and between MCMC chains; values below 1.01 indicate that the
chains explored the same posterior region. ESS is effective sample size: the number of independent
draws represented by correlated chain draws. Divergences indicate failures to follow posterior
geometry and must be zero.


In [ ]:
def diagnostic_table(idata: az.InferenceData) -> pd.DataFrame:
    table = az.summary(idata, kind="diagnostics", round_to="none")
    columns = ["ess_bulk", "ess_tail", "r_hat"]
    table = table.reindex(columns=columns)
    table[columns] = table[columns].apply(pd.to_numeric, errors="coerce")
    return table


def fit_gate(label: str, idata: az.InferenceData, ess_min: float) -> tuple[dict, pd.DataFrame]:
    table = diagnostic_table(idata)
    divergences = int(np.asarray(idata.sample_stats["diverging"]).sum())
    finite = bool(np.isfinite(table.to_numpy(float)).all())
    result = {
        "fit": label,
        "max_rhat": float(table["r_hat"].max()),
        "min_ess_bulk": float(table["ess_bulk"].min()),
        "min_ess_tail": float(table["ess_tail"].min()),
        "divergences": divergences,
    }
    result["diagnostics_ok"] = bool(
        finite
        and result["max_rhat"] < bc.RHAT_MAX
        and result["min_ess_bulk"] > ess_min
        and result["min_ess_tail"] > ess_min
        and divergences == 0
    )
    return result, table


ESS_THRESHOLD = bc.ESS_MIN if IS_FULL else PILOT_ESS_MIN
primary_diagnostic, primary_parameter_diagnostics = fit_gate(
    "A-freq primary Hann", idata_primary, ESS_THRESHOLD
)
headline_names = ["beta_bar", "delta_O", "delta_P", "tau", "sigma_harm", "sigma_bg", "sigma"]
headline_diagnostics = primary_parameter_diagnostics.reindex(headline_names)
HEADLINE_OK = bool(
    headline_diagnostics.notna().all().all()
    and (headline_diagnostics["r_hat"] < bc.RHAT_MAX).all()
    and (headline_diagnostics["ess_bulk"] > ESS_THRESHOLD).all()
    and (headline_diagnostics["ess_tail"] > ESS_THRESHOLD).all()
    and primary_diagnostic["divergences"] == 0
)

beta_draws = np.asarray(
    idata_primary.posterior["beta_bar"],
    dtype=float,
).ravel()

P_ATTENUATION = float(
    np.mean(beta_draws <= QUALITY_DIFF_ATTENUATION)
)

P_EQUIVALENT = float(
    np.mean(np.abs(beta_draws) <= QUALITY_DIFF_ROPE)
)

P_ANY_ATTENUATION = float(
    np.mean(beta_draws < 0.0)
)

print("P(difference <= -0.20 | data) =", round(P_ATTENUATION, 6))
print("P(abs(difference) <= 0.10 | data) =", round(P_EQUIVALENT, 6))
print("P(difference < 0 | data) =", round(P_ANY_ATTENUATION, 6))

summary_rows = []
for name, values in {
    "beta_bar (mean lock-control quality difference)": beta_draws,
    "delta_O (overlap mitigation)": np.asarray(idata_primary.posterior["delta_O"]).ravel(),
    "delta_P (log patch-size slope)": np.asarray(idata_primary.posterior["delta_P"]).ravel(),
}.items():
    low, high = np.quantile(values, [0.025, 0.975])
    summary_rows.append(
        {
            "parameter": name,
            "median": float(np.median(values)),
            "eti_low": float(low),
            "eti_high": float(high),
            "sd": float(np.std(values)),
        }
    )
scientific_summary = pd.DataFrame(summary_rows)

display(scientific_summary.round(6))
print(primary_diagnostic)
print("headline estimands gate:", HEADLINE_OK)
print("P(beta_bar < 0 | data) =", round(P_ANY_ATTENUATION, 6))
print("\nWorst R-hat")
display(primary_parameter_diagnostics.sort_values("r_hat", ascending=False).head(12))
print("\nLowest bulk ESS")
display(primary_parameter_diagnostics.sort_values("ess_bulk").head(12))


## 6. Required method checks

### 6.1 Synthetic parameter recovery

This small synthetic dataset is generated from known coefficients and then fitted with the same
model. A 95 percent interval must cover each known scientific coefficient, and those coefficients
must converge. The numbers below are validation only and must never be reported as findings about
Chronos.


In [ ]:
RECOVERY_TABLE_PATH = OUTPUT_ROOT / "parameter_recovery.parquet"


def zero_sum(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    return values - values.mean()


def simulate_parameter_recovery(frame: pd.DataFrame, seed: int = SEED):
    rng = np.random.default_rng(seed)
    simulated = frame[frame["bg_id"] < 3].copy().reset_index(drop=True)
    config_code, config_levels = codes(simulated["model"].astype(str))
    harmonic_code, harmonic_levels = codes(simulated["f_lock"].map(lambda x: f"{x:.9f}"))
    background_code, background_levels = codes(
        simulated["generator"].astype(str) + "#" + simulated["bg_id"].astype(str)
    )
    truths = {"beta_bar": -0.40, "delta_O": 0.25, "delta_P": -0.15}
    configuration_mean = (
        truths["beta_bar"]
        + truths["delta_O"] * overlap_scaled(simulated, config_levels)
        + truths["delta_P"] * log_patch_centred(simulated, config_levels)
    )
    beta_truth = configuration_mean + zero_sum(rng.normal(0.0, 0.08, len(config_levels)))
    harmonic_truth = zero_sum(rng.normal(0.0, 0.08, len(harmonic_levels)))
    background_truth = zero_sum(rng.normal(0.0, 0.06, len(background_levels)))
    location = (
        beta_truth[config_code]
        + harmonic_truth[harmonic_code]
        + background_truth[background_code]
    )
    simulated["recovery_d_quality"] = (
        location
        + 0.20 * rng.standard_t(
            NU,
            size=len(location),
        )
    )
    return simulated, truths


if IS_FULL:
    if artifact_is_valid(RECOVERY_TABLE_PATH):
        recovery_table = pd.read_parquet(RECOVERY_TABLE_PATH)
    else:
        recovery_frame, recovery_truths = simulate_parameter_recovery(observations)
        recovery_model = model_A_freq(recovery_frame, response_column="recovery_d_quality")
        recovery_idata = fit_or_load(
            "04_A_freq_parameter_recovery",
            recovery_model,
            draws=RECOVERY_DRAWS,
            tune=RECOVERY_TUNE,
            target_accept=0.95,
        )
        recovery_diagnostic_table = diagnostic_table(recovery_idata)
        recovery_divergences = int(np.asarray(recovery_idata.sample_stats["diverging"]).sum())
        rows = []
        for parameter, truth in recovery_truths.items():
            values = np.asarray(recovery_idata.posterior[parameter], dtype=float).ravel()
            low, high = np.quantile(values, [0.025, 0.975])
            diagnostic = recovery_diagnostic_table.loc[parameter]
            rows.append(
                {
                    "parameter": parameter,
                    "truth": truth,
                    "median": float(np.median(values)),
                    "eti_low": float(low),
                    "eti_high": float(high),
                    "covered": bool(low <= truth <= high),
                    "ess_bulk": float(diagnostic["ess_bulk"]),
                    "ess_tail": float(diagnostic["ess_tail"]),
                    "r_hat": float(diagnostic["r_hat"]),
                    "divergences": recovery_divergences,
                }
            )
        recovery_table = pd.DataFrame(rows)
        cp.atomic_parquet(RECOVERY_TABLE_PATH, recovery_table)
        record_artifact(RECOVERY_TABLE_PATH)
        del recovery_frame, recovery_model, recovery_idata
        gc.collect()
    RECOVERY_OK = bool(
        recovery_table["covered"].all()
        and (recovery_table["r_hat"] < bc.RHAT_MAX).all()
        and (recovery_table["ess_bulk"] > bc.ESS_MIN).all()
        and (recovery_table["ess_tail"] > bc.ESS_MIN).all()
        and (recovery_table["divergences"] == 0).all()
    )
else:
    recovery_table = pd.DataFrame()
    RECOVERY_OK = False
    print("PILOT: parameter recovery is deferred to FULL mode")

display(recovery_table)
print("PARAMETER RECOVERY GATE:", "PASS" if RECOVERY_OK else "FAIL OR PILOT")


### 6.2 Posterior-predictive check

PPC means posterior-predictive check: simulated datasets from the fitted model are compared with
the observed configuration and generator means. It asks whether the model can reproduce the data
patterns it claims to explain. At least 90 percent of the required strata must lie inside their
95 percent replicated intervals.


In [ ]:
PPC_PATH = OUTPUT_ROOT / "posterior_predictive_check.parquet"


def stack_samples(array: xr.DataArray, level_dimension: str | None = None) -> np.ndarray:
    dimensions = ["chain", "draw"] + ([level_dimension] if level_dimension else [])
    values = np.asarray(array.transpose(*dimensions), dtype=float)
    if level_dimension:
        return values.reshape(-1, values.shape[-1])
    return values.reshape(-1)


def codes_against(values, levels, label: str) -> np.ndarray:
    mapping = {str(level): index for index, level in enumerate(levels)}
    result = np.asarray([mapping.get(str(value), -1) for value in values], dtype=int)
    if np.any(result < 0):
        raise ValueError(f"{label} contains levels absent from posterior coordinates")
    return result


def manual_ppc(idata: az.InferenceData, frame: pd.DataFrame, response_column: str) -> pd.DataFrame:
    posterior = idata.posterior
    config_levels = list(map(str, posterior.coords["config"].values))
    harmonic_levels = list(map(str, posterior.coords["harmonic"].values))
    background_levels = list(map(str, posterior.coords["background"].values))
    config_code = codes_against(frame["model"].astype(str), config_levels, "model")
    harmonic_code = codes_against(
        frame["f_lock"].map(lambda x: f"{x:.9f}"), harmonic_levels, "harmonic"
    )
    background_code = codes_against(
        frame["generator"].astype(str) + "#" + frame["bg_id"].astype(str),
        background_levels,
        "background",
    )
    beta = stack_samples(posterior["beta"], "config")
    u_harm = stack_samples(posterior["u_harm"], "harmonic")
    u_bg = stack_samples(posterior["u_bg"], "background")
    sigma = stack_samples(posterior["sigma"])
    n_samples = len(sigma)
    selected = np.linspace(0, n_samples - 1, min(PPC_DRAWS, n_samples)).round().astype(int)
    rng = np.random.default_rng(SEED + 501)

    strata = []
    for column in ("model", "generator"):
        for level, positions in frame.groupby(column, observed=True).indices.items():
            strata.append((f"{column}={level}", np.asarray(positions, dtype=int)))
    replicated = {name: [] for name, _ in strata}

    for draw_number, sample in enumerate(selected, start=1):
        location = (
            beta[sample, config_code]
            + u_harm[sample, harmonic_code]
            + u_bg[sample, background_code]
        )
        replicate = (
            location
            + sigma[sample]
            * rng.standard_t(
                NU,
                size=location.shape[0],
            )
        )
        for name, positions in strata:
            replicated[name].append(float(np.mean(replicate[positions])))
        if draw_number == 1 or draw_number % 50 == 0 or draw_number == len(selected):
            print(f"PPC draw {draw_number}/{len(selected)}")

    observed = frame[response_column].to_numpy(float)
    rows = []
    for name, positions in strata:
        values = np.asarray(replicated[name], dtype=float)
        low, high = np.quantile(values, [0.025, 0.975])
        observed_mean = float(np.mean(observed[positions]))
        rows.append(
            {
                "stratum": name,
                "n": len(positions),
                "observed_mean": observed_mean,
                "rep_low": float(low),
                "rep_high": float(high),
                "rep_centre": float(np.median(values)),
                "observed_minus_rep": observed_mean - float(np.median(values)),
                "ppc_ok": bool(low <= observed_mean <= high),
            }
        )
    return pd.DataFrame(rows)


if IS_FULL:
    if artifact_is_valid(PPC_PATH):
        ppc_table = pd.read_parquet(PPC_PATH)
    else:
        ppc_table = manual_ppc(idata_primary, observations, "pred_hann_d_quality")
        cp.atomic_parquet(PPC_PATH, ppc_table)
        record_artifact(PPC_PATH)
    required_ppc = {
        *(f"model={model}" for model in sorted(observations["model"].unique())),
        *(f"generator={generator}" for generator in sorted(observations["generator"].unique())),
    }
    PPC_OK = bc.ppc_gate(ppc_table, required_ppc, PPC_MIN_COVERAGE)
else:
    ppc_table = pd.DataFrame()
    PPC_OK = False
    print("PILOT: PPC is deferred to FULL mode")

display(ppc_table.sort_values("observed_minus_rep", key=lambda x: x.abs(), ascending=False).head(20) if not ppc_table.empty else ppc_table)
print("PPC GATE:", "PASS" if PPC_OK else "FAIL OR PILOT")


### 6.3 Prior and window sensitivity

Sensitivity analysis repeats the fit under reasonable alternative choices. It protects the
conclusion from being an accident of one prior width or one spectral window. FULL mode fits prior
scales 0.25 and 1.0 under Hann, plus the rectangular-window response at scale 0.5. The support and
equivalence probabilities may vary by at most 0.10.


#### 6.3.1 Hann, prior scale 0.25


In [ ]:
if IS_FULL:
    idata_scale025 = fit_or_load(
        "04_A_freq_hann_prior025",
        model_A_freq(observations, prior_scale=0.25),
    )
else:
    idata_scale025 = None
    print("PILOT: skipped")


#### 6.3.2 Hann, prior scale 1.0


In [ ]:
if IS_FULL:
    idata_scale100 = fit_or_load(
        "04_A_freq_hann_prior100",
        model_A_freq(observations, prior_scale=1.0),
    )
else:
    idata_scale100 = None
    print("PILOT: skipped")


#### 6.3.3 Rectangular window, prior scale 0.5


In [ ]:
if IS_FULL:
    idata_rectangular = fit_or_load(
        "04_A_freq_rectangular_primary_prior",
        model_A_freq(
            observations,
            response_column="pred_rectangular_d_quality",
            prior_scale=0.5,
        ),
    )
else:
    idata_rectangular = None
    print("PILOT: skipped")


#### 6.3.4 Evaluate the sensitivity gate


In [ ]:
def sensitivity_row(label: str, idata: az.InferenceData) -> dict:
    diagnostic, _ = fit_gate(label, idata, bc.ESS_MIN)
    beta = np.asarray(idata.posterior["beta_bar"], dtype=float).ravel()
    return {
        "specification": label,
        "median_quality_difference": float(np.median(beta)),
        "p_difference_le_minus020": float(np.mean(beta <= QUALITY_DIFF_ATTENUATION)),
        "p_practical_equivalence": float(np.mean(np.abs(beta) <= QUALITY_DIFF_ROPE)),
        **diagnostic,
    }


if IS_FULL:
    sensitivity_table = pd.DataFrame(
        [
            sensitivity_row("Hann scale 0.5 primary", idata_primary),
            sensitivity_row("Hann scale 0.25", idata_scale025),
            sensitivity_row("Hann scale 1.0", idata_scale100),
            sensitivity_row("Rectangular scale 0.5", idata_rectangular),
        ]
    )
    sensitivity_probability_ok = bc.sensitivity_gate(
        sensitivity_table,
        ("p_difference_le_minus020", "p_practical_equivalence"),
        SENSITIVITY_MAX_SPREAD,
    )
    sensitivity_diagnostics_ok = bool(sensitivity_table["diagnostics_ok"].all())
    SENSITIVITY_OK = bool(sensitivity_probability_ok and sensitivity_diagnostics_ok)
    SENSITIVITY_PATH = OUTPUT_ROOT / "sensitivity.parquet"
    cp.atomic_parquet(SENSITIVITY_PATH, sensitivity_table)
    record_artifact(SENSITIVITY_PATH)
else:
    sensitivity_table = pd.DataFrame()
    sensitivity_probability_ok = False
    sensitivity_diagnostics_ok = False
    SENSITIVITY_OK = False

display(sensitivity_table)
print("probability spread gate:", sensitivity_probability_ok)
print("sensitivity diagnostics:", sensitivity_diagnostics_ok)
print("SENSITIVITY GATE:", "PASS" if SENSITIVITY_OK else "FAIL OR PILOT")


## 7. Trace plots

Each line is one MCMC chain. Good mixing appears as overlapping stationary bands, rather than
chains occupying persistently different levels. These plots support diagnosis but do not replace
the numerical gates.


In [ ]:
trace_variables = ["beta_bar", "delta_O", "delta_P", "tau", "sigma_harm", "sigma_bg", "sigma"]
posterior = idata_primary.posterior
figure, axes = plt.subplots(len(trace_variables), 1, figsize=(12, 2.0 * len(trace_variables)), sharex=True)
for axis, variable in zip(axes, trace_variables):
    values = np.asarray(posterior[variable].transpose("chain", "draw"), dtype=float)
    for chain_index, chain_values in enumerate(values):
        axis.plot(chain_values, lw=0.65, alpha=0.75, label=f"chain {chain_index}")
    axis.set_ylabel(variable)
axes[0].legend(ncol=min(CHAINS, 4), fontsize=8, loc="upper right")
axes[-1].set_xlabel("retained draw")
figure.suptitle(f"Model A-freq primary traces, {RUN_MODE}", y=1.002)
figure.tight_layout()
TRACE_PATH = FIGURE_ROOT / "A_freq_primary_trace.png"
figure.savefig(TRACE_PATH, dpi=140, bbox_inches="tight")
record_artifact(TRACE_PATH)
plt.show()


## 8. Final fail-closed H1-freq verdict

The posterior is the distribution of plausible effects after combining the prior with the
observations. A failed validation gate produces `NOT REPORTABLE`; it is never converted into
`FALSE`. `INCONCLUSIVE` means the analysis is valid but posterior evidence is insufficient for
either the support or equivalence rule.


In [ ]:
PRIMARY_DIAGNOSTICS_OK = bool(primary_diagnostic["diagnostics_ok"] and HEADLINE_OK)
H1_FREQ_GATE_OK = bool(
    IS_FULL
    and TRUTH_ESTIMATOR_OK
    and PRIMARY_DIAGNOSTICS_OK
    and RECOVERY_OK
    and PPC_OK
    and SENSITIVITY_OK
)
h1_freq_project_verdict = bc.three_way_verdict(
    support_probability=P_ATTENUATION,
    refute_probability=P_EQUIVALENT,
    gate_ok=H1_FREQ_GATE_OK,
    cutoff=POSTERIOR_CUTOFF,
)


def tfi_label(project_verdict: str) -> str:
    return {
        "supported": "TRUE",
        "refuted": "FALSE",
        "inconclusive": "INCONCLUSIVE",
        "NOT REPORTABLE": "NOT REPORTABLE",
    }[project_verdict]


gate_table = pd.DataFrame(
    [
        {"gate": "FULL mode", "passed": IS_FULL},
        {"gate": "true-future frequency estimator", "passed": TRUTH_ESTIMATOR_OK},
        {"gate": "primary global and headline convergence", "passed": PRIMARY_DIAGNOSTICS_OK},
        {"gate": "synthetic parameter recovery", "passed": RECOVERY_OK},
        {"gate": "posterior-predictive check", "passed": PPC_OK},
        {"gate": "prior and window sensitivity", "passed": SENSITIVITY_OK},
    ]
)
verdict_table = pd.DataFrame(
    [
        {
            "hypothesis": "H1-freq",
            "model": "A-freq v3 Student-t quality difference",
            "gate_ok": H1_FREQ_GATE_OK,
            "p_difference_le_minus020": P_ATTENUATION,
            "p_practical_equivalence": P_EQUIVALENT,
            "project_verdict": h1_freq_project_verdict,
            "TFI": tfi_label(h1_freq_project_verdict),
            "rule": "P(beta_bar <= -0.20) >= 0.95; refute if P(abs(beta_bar) <= 0.10) >= 0.95",
        }
    ]
)

GATE_PATH = OUTPUT_ROOT / "A_freq_gate_table.parquet"
VERDICT_PATH = OUTPUT_ROOT / "A_freq_final_verdict.parquet"
FINAL_JSON_PATH = OUTPUT_ROOT / "A_freq_final_verdict.json"
cp.atomic_parquet(GATE_PATH, gate_table)
cp.atomic_parquet(VERDICT_PATH, verdict_table)
cp.atomic_json(
    FINAL_JSON_PATH,
    {
        "analysis_fingerprint": ANALYSIS_FINGERPRINT,
        "core_design_fingerprint": CORE_DESIGN_FINGERPRINT,
        "run_mode": RUN_MODE,
        "gate_ok": H1_FREQ_GATE_OK,
        "pilot_route_ok": bool(TRUTH_ESTIMATOR_OK and PRIMARY_DIAGNOSTICS_OK),
        "truth_estimator_ok": TRUTH_ESTIMATOR_OK,
        "primary_diagnostics_ok": PRIMARY_DIAGNOSTICS_OK,
        "verdict": json.loads(verdict_table.to_json(orient="records"))[0],
        "primary_diagnostic": primary_diagnostic,
    },
)
for path in (GATE_PATH, VERDICT_PATH, FINAL_JSON_PATH):
    record_artifact(path)

display(gate_table)
display(verdict_table)
print("=" * 78)
print("FINAL H1-FREQ:", verdict_table.iloc[0]["TFI"])
print("=" * 78)
if not IS_FULL:
    if TRUTH_ESTIMATOR_OK and PRIMARY_DIAGNOSTICS_OK:
        print("PILOT PASS: switch RUN_MODE to 'FULL', reconnect to a clean runtime, and Run all.")
    else:
        print("PILOT FAIL: do not launch FULL. Inspect the failed estimator or convergence gate.")
elif not H1_FREQ_GATE_OK:
    print("FULL completed but at least one scientific-validity gate failed. See gate_table.")
else:
    print("Reportable FULL verdict produced.")
print("all outputs:", OUTPUT_ROOT)


## 9. Scope and interpretation

- This v3 implements the latest quality-difference proposal, rather than the earlier v2 log-ratio.
- Previous A posteriors and validation outputs cannot be reused. Compatible frequency pair tables
  can be reused because both per-role qualities are available and the new difference is recomputed.
- `-0.20` is an absolute quality difference. It does not mean a relative 20 percent reduction.
- `delta_O` and `delta_P` remain descriptive here. M1/A-prime requires separately fitted predictor
  variants and a reliable LOO comparison under this response and likelihood.
- PILOT validates a route to FULL; it cannot supply an empirical H1 verdict. A failed gate means
  NOT REPORTABLE. Passing computational checks alone does not establish model adequacy.
- B, C, D1 and D2 are outside this notebook. No report source is edited.


### 9.1 Inspect the response and posterior scales

Use these diagnostics to interpret a failed pilot before considering another run.

In [ ]:
d_quality = observations["pred_hann_d_quality"].astype(float)

print("QUANTILI DELLA RISPOSTA")
display(
    d_quality.quantile(
        [0.0, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 1.0]
    ).rename("d_quality").to_frame()
)

print("quota |d_quality| <= 0.01:",
      float((d_quality.abs() <= 0.01).mean()))
print("min:", float(d_quality.min()))
print("max:", float(d_quality.max()))

print("\nSCALE GERARCHICHE")
for variable in ("tau", "sigma_harm", "sigma_bg", "sigma"):
    values = np.asarray(idata_primary.posterior[variable], dtype=float).ravel()
    low, median, high = np.quantile(values, [0.025, 0.5, 0.975])
    diagnostic = primary_parameter_diagnostics.loc[variable]
    print(
        f"{variable:12s} median={median:.6g} "
        f"95%=[{low:.6g}, {high:.6g}] "
        f"ESS={diagnostic['ess_bulk']:.1f} "
        f"R-hat={diagnostic['r_hat']:.4f}"
    )

print("\nBETA_BAR PER CATENA")
for chain in idata_primary.posterior.coords["chain"].values:
    values = np.asarray(
        idata_primary.posterior["beta_bar"].sel(chain=chain), dtype=float
    ).ravel()
    print(
        f"chain {chain}: median={np.median(values):.5f}, "
        f"95%={np.quantile(values, [0.025, 0.975])}"
    )